# MAV Flight Physics: Frames, Forces, Moments, and Equations of Motion

This document derives the six-degree-of-freedom rigid-body flight model implemented by
`Dynamics` and `ForcesAndMoments`. Its main purpose is to make the coordinate-frame
bookkeeping explicit:

- where each force and moment is resolved,
- what point the moments are taken about,
- how gravity enters the body-axis equations,
- why the translational equations contain cross-coupling terms, and
- how to rotate forces or accelerations back into an inertial frame.

The implementation is a standard fixed-wing MAV model with 12 states, Euler-angle
attitude, aerodynamic coefficient models, propeller thrust, and rigid-body rotational
dynamics.

---



## 1. Synopsis

The simulation uses two principal frames:

1. **Inertial/navigation frame $n$: North-East-Down (NED)**
2. **Body frame $b$: $x_b$ forward, $y_b$ right, $z_b$ down**

The state velocities $[u,v,w]^T$, all calculated forces
$[F_x,F_y,F_z]^T$, angular rates $[p,q,r]^T$, and moments
$[\ell,m,n]^T$ are expressed in the **body frame**.

The position $[p_n,p_e,p_d]^T$ is expressed in the **NED inertial frame**.

Gravity is:

```math
\mathbf F_g^n =
\begin{bmatrix}
0\\0\\mg
\end{bmatrix},
```

because the third NED axis points downward. `ForcesAndMoments.calculate()` rotates this
gravity vector into the body frame and then adds it to the body-frame aerodynamic and
propulsive forces:

```math
\mathbf F_{\text{total}}^b
=
\mathbf F_g^b
+\mathbf F_a^b
+\mathbf F_p^b.
```

The simulator then integrates Newton's second law in the rotating body frame:

```math
\dot{\mathbf v}^{\,b}
=
\frac{1}{m}\mathbf F_{\text{total}}^b
-\boldsymbol\omega^b\times\mathbf v^b.
```

**the forces do not need to be rotated into the inertial frame before the
body-axis velocity derivative is calculated**. The cross-product term is what makes
Newton's law valid in the rotating body frame.

If an inertial/NED force or acceleration is needed, use

```math
\mathbf F_{\text{total}}^n
=
R_b^n\mathbf F_{\text{total}}^b,
\qquad
\mathbf a^n
=
\frac{1}{m}R_b^n\mathbf F_{\text{total}}^b.
```

Because gravity is already included in `forces`, the second expression gives total
inertial acceleration, including gravitational acceleration.

---



## 2. State and control definitions

The 12-state vector is

```math
\mathbf x =
\begin{bmatrix}
p_n & p_e & p_d &
u & v & w &
\phi & \theta & \psi &
p & q & r
\end{bmatrix}^T.
```

| State | Frame | Meaning | Positive direction |
|---|---|---|---|
| $p_n$ | NED | North position | North |
| $p_e$ | NED | East position | East |
| $p_d$ | NED | Down position | Down |
| $u$ | Body | Forward velocity | $+x_b$, through the nose |
| $v$ | Body | Lateral velocity | $+y_b$, through the right wing |
| $w$ | Body | Vertical body velocity | $+z_b$, through the belly |
| $\phi$ | Euler angle | Roll | Right wing down |
| $\theta$ | Euler angle | Pitch | Nose up |
| $\psi$ | Euler angle | Yaw/heading | Clockwise when viewed from above in NED |
| $p$ | Body | Roll rate | About $+x_b$ |
| $q$ | Body | Pitch rate | About $+y_b$ |
| $r$ | Body | Yaw rate | About $+z_b$ |

Since $p_d$ is positive downward, geometric altitude is normally

```math
h=-p_d.
```

The control vector is unpacked as

```math
\boldsymbol\delta =
\begin{bmatrix}
\delta_e & \delta_t & \delta_a & \delta_r
\end{bmatrix}^T,
```

where the entries are elevator, throttle, aileron, and rudder commands.

---



## 3. Reference frames and the direction-cosine matrix

The attitude uses the standard 3-2-1 yaw-pitch-roll Euler sequence:

1. yaw by $\psi$,
2. pitch by $\theta$,
3. roll by $\phi$.

Let

```math
c_\phi=\cos\phi,\quad s_\phi=\sin\phi,
```

with analogous notation for $\theta$ and $\psi$. The body-to-NED rotation matrix
consistent with the gravity and Euler-rate equations in the code is

```math
R_b^n(\phi,\theta,\psi)=
\begin{bmatrix}
c_\theta c_\psi &
s_\phi s_\theta c_\psi-c_\phi s_\psi &
c_\phi s_\theta c_\psi+s_\phi s_\psi\\
c_\theta s_\psi &
s_\phi s_\theta s_\psi+c_\phi c_\psi &
c_\phi s_\theta s_\psi-s_\phi c_\psi\\
-s_\theta &
s_\phi c_\theta &
c_\phi c_\theta
\end{bmatrix}.
```

It maps body-resolved vector components into NED components:

```math
\mathbf a^n=R_b^n\mathbf a^b.
```

Because this is an orthogonal rotation matrix,

```math
R_n^b=(R_b^n)^T,
```

and a NED vector is rotated into body components using

```math
\mathbf a^b=R_n^b\mathbf a^n=(R_b^n)^T\mathbf a^n.
```

`body_to_inertial(phi, theta, psi)` should return exactly $R_b^n$, subject only to
equivalent algebraic arrangement.



## 4. Position kinematics

The velocity states are body components, but the position states are NED components.
The position derivative is therefore

```math
\dot{\mathbf p}^{\,n}
=
R_b^n\mathbf v^b,
\qquad
\mathbf v^b=
\begin{bmatrix}u\\v\\w\end{bmatrix}.
```

This is implemented by

```python
position_dot = body_to_inertial(phi, theta, psi) @ np.array([[u], [v], [w]])
```

No force is involved in this line. It only changes the coordinate representation of
the aircraft velocity so that it can update NED position.

---



## 5. Newton's second law in a rotating body frame

Newton's second law in its standard form only applies in an inertial frame:

```math
\mathbf F_{\text{total}}^n
=
m\frac{d\mathbf v^n}{dt}.
```

The simulation stores velocity in body components instead. For any vector
$\mathbf a$, the transport theorem gives

```math
\left(\frac{d\mathbf a}{dt}\right)_n
=
\left(\frac{d\mathbf a}{dt}\right)_b
+\boldsymbol\omega^b\times\mathbf a^b.
```

Applying the theorem to translational velocity gives

```math
\mathbf F_{\text{total}}^b
=
m\left(
\dot{\mathbf v}^{\,b}
+\boldsymbol\omega^b\times\mathbf v^b
\right),
```

so

```math
\dot{\mathbf v}^{\,b}
=
\frac{\mathbf F_{\text{total}}^b}{m}
-\boldsymbol\omega^b\times\mathbf v^b.
```

With

```math
\boldsymbol\omega^b=
\begin{bmatrix}p\\q\\r\end{bmatrix},
\qquad
\mathbf v^b=
\begin{bmatrix}u\\v\\w\end{bmatrix},
```

the cross product is

```math
\boldsymbol\omega^b\times\mathbf v^b
=
\begin{bmatrix}
qw-rv\\
ru-pw\\
pv-qu
\end{bmatrix}.
```

Therefore,

```math
\begin{aligned}
\dot u &= rv-qw+\frac{F_x}{m},\\
\dot v &= pw-ru+\frac{F_y}{m},\\
\dot w &= qu-pv+\frac{F_z}{m}.
\end{aligned}
```

These are the three equations in `Dynamics.derivative()`.

### Why this is still Newton's law

The terms $rv-qw$, $pw-ru$, and $qu-pv$ are not additional physical forces.
They appear because the body axes rotate while the vector is being differentiated.
There are two equivalent and valid approaches:

**Approach A — the method used by this simulation**

```math
\dot{\mathbf v}^{\,b}
=
\frac{\mathbf F^b}{m}
-\boldsymbol\omega^b\times\mathbf v^b.
```

**Approach B — perform the acceleration calculation in NED**

```math
\mathbf a^n
=
\frac{\mathbf F^n}{m}
=
\frac{1}{m}R_b^n\mathbf F^b.
```

Do not rotate the force into NED and then also apply the body-frame cross-product
correction. That would mix the two formulations and double-count the rotating-frame
effect.

---



## 6. Where the forces and moments act

`ForcesAndMoments.calculate()` returns a total force and total moment:

```math
\mathbf F^b =
\begin{bmatrix}F_x\\F_y\\F_z\end{bmatrix},
\qquad
\mathbf M^b =
\begin{bmatrix}\ell\\m\\n\end{bmatrix}.
```

All six components are **resolved along the body axes**.

The rigid-body equations implicitly make the following reference-point assumptions:

- Translational force is treated as a net resultant acting on the center of mass.
- Gravity acts through the center of mass and creates no direct moment.
- Aerodynamic moments are moments about the aerodynamic reference point represented
  by the coefficient data; for the equations to be used directly as written, that
  reference should be the aircraft center of gravity.
- Propeller thrust acts along $+x_b$. No thrust-offset moment is calculated.
- Propeller reaction torque is included directly in the roll-moment expression.
- No individual center-of-pressure location or force lever arm is modeled.

Thus, the code does not determine aerodynamic moment by computing
$\mathbf r\times\mathbf F$ from a wing force location. Instead, the rolling,
pitching, and yawing moment coefficients already represent the net aerodynamic
moments about the chosen reference point.

If the coefficient database is referenced somewhere other than the center of gravity,
the moment must be transferred to the center of gravity:

```math
\mathbf M_{\mathrm{CG}}^b
=
\mathbf M_{\mathrm{ref}}^b
+\mathbf r_{\mathrm{CG}\rightarrow\mathrm{ref}}^b
\times\mathbf F^b.
```

The direction of $\mathbf r_{\mathrm{CG}\rightarrow\mathrm{ref}}^b$ matters: it runs
from the new moment point, the CG, to the original reference point.

---



## 7. Air data

The current implementation calculates

```math
V_a=\sqrt{u^2+v^2+w^2},
```

with a small positive floor to avoid division by zero. It then calculates

```math
\alpha=\operatorname{atan2}(w,u),
\qquad
\beta=\sin^{-1}\left(\frac{v}{V_a}\right).
```

Under the $x_b$-forward, $y_b$-right, $z_b$-down convention:

- positive $\alpha$ corresponds to positive $w$,
- positive $\beta$ corresponds to positive $v$, or airflow-relative motion toward
  the right in the velocity-vector convention used here.

### Important wind assumption

The code uses body velocity directly as air-relative velocity. Consequently,
$V_a$, $\alpha$, and $\beta$ are physically correct only when the surrounding
air is stationary in the NED frame.

With wind, first calculate relative air velocity:

```math
\mathbf v_{\text{air}}^b
=
\mathbf v_{\text{ground}}^b
-R_n^b\mathbf v_{\text{wind}}^n,
```

and then calculate $V_a$, $\alpha$, and $\beta$ from
$\mathbf v_{\text{air}}^b$.

---



## 8. Aerodynamic lift and drag

Define dynamic pressure as

```math
\bar q=\frac{1}{2}\rho V_a^2.
```

### Lift coefficient and stall blending

The attached-flow lift model is

```math
C_{L,\text{linear}}=C_{L_0}+C_{L_\alpha}\alpha.
```

The code blends it with a post-stall flat-plate approximation using

```math
\sigma(\alpha)=
\frac{
1+e^{-M(\alpha-\alpha_0)}+e^{M(\alpha+\alpha_0)}
}{
\left(1+e^{-M(\alpha-\alpha_0)}\right)
\left(1+e^{M(\alpha+\alpha_0)}\right)
},
```

where $M$ is `stall_slope` and $\alpha_0$ is `alpha0`. The blended coefficient is

```math
C_L
=
(1-\sigma)(C_{L_0}+C_{L_\alpha}\alpha)
+\sigma\left(2\,\operatorname{sign}(\alpha)\sin^2\alpha\cos\alpha\right).
```

For small angles, $\sigma$ is intended to be near zero and the linear lift curve
dominates. Near and beyond stall, the nonlinear flat-plate expression increasingly
dominates.

### Drag coefficient

The drag coefficient is modeled as parasitic plus induced drag:

```math
C_D
=
C_{D_p}
+\frac{(C_{L_0}+C_{L_\alpha}\alpha)^2}
{\pi e\,AR}.
```

Here $e$ is the Oswald efficiency factor and $AR$ is wing aspect ratio.

The induced-drag term currently uses the linear lift coefficient even when the
stall-blended $C_L$ is active. This is the behavior of the code, although a
high-angle-of-attack model may instead use the blended lift coefficient or a separate
post-stall drag model.

---



## 9. Rotation of lift and drag into body axes

Lift and drag are naturally defined relative to the air-velocity, or wind, axes:

- drag acts opposite the flight direction,
- lift acts perpendicular to the flight direction,
- side force acts laterally.

In the longitudinal plane, take

```math
D=\bar q S C_D,
\qquad
L=\bar q S C_L.
```

With body $z_b$ positive down, the wind-axis force vector at zero sideslip is

```math
\begin{bmatrix}
-D\\
0\\
-L
\end{bmatrix}.
```

Rotating that vector through angle of attack into the body $x_b$-$z_b$ plane gives

```math
\begin{aligned}
X &= -D\cos\alpha+L\sin\alpha,\\
Z &= -D\sin\alpha-L\cos\alpha.
\end{aligned}
```

After division by $\bar qS$, the corresponding body-axis coefficients are

```math
\boxed{
C_X=-C_D\cos\alpha+C_L\sin\alpha
}
```

and

```math
\boxed{
C_Z=-C_D\sin\alpha-C_L\cos\alpha.
}
```

The code applies the same rotation to pitch-rate and elevator derivatives:

```math
\begin{aligned}
C_{X_q} &=-C_{D_q}\cos\alpha+C_{L_q}\sin\alpha,\\
C_{X_{\delta_e}} &=-C_{D_{\delta_e}}\cos\alpha
                    +C_{L_{\delta_e}}\sin\alpha,\\
C_{Z_q} &=-C_{D_q}\sin\alpha-C_{L_q}\cos\alpha,\\
C_{Z_{\delta_e}} &=-C_{D_{\delta_e}}\sin\alpha
                    -C_{L_{\delta_e}}\cos\alpha.
\end{aligned}
```

Wind rotation for the force model: **lift and drag are converted
from air-velocity-aligned directions into body-axis $X$ and $Z$ components**.

The implementation does not perform a complete three-dimensional wind-to-body
rotation using both $\alpha$ and $\beta$. Instead, it uses the longitudinal
$\alpha$ rotation for $X$ and $Z$, then models the body $Y$ force separately
with lateral stability derivatives. This is common for a coefficient-based small-MAV
model, especially when sideslip remains modest.

---



## 10. Complete aerodynamic force

The nondimensional body-axis force coefficients used in the code are

```math
\begin{aligned}
C_X^\ast &=
C_X+C_{X_q}\frac{c}{2V_a}q+C_{X_{\delta_e}}\delta_e,\\
C_Y^\ast &=
C_{Y_0}+C_{Y_\beta}\beta
+C_{Y_p}\frac{b}{2V_a}p
+C_{Y_r}\frac{b}{2V_a}r
+C_{Y_{\delta_a}}\delta_a
+C_{Y_{\delta_r}}\delta_r,\\
C_Z^\ast &=
C_Z+C_{Z_q}\frac{c}{2V_a}q+C_{Z_{\delta_e}}\delta_e.
\end{aligned}
```

Here $S$ is wing planform area, $b$ is wingspan, and $c$ is mean aerodynamic
chord. The aerodynamic force returned in body components is

```math
\boxed{
\mathbf F_a^b
=
\bar qS
\begin{bmatrix}
C_X^\ast\\
C_Y^\ast\\
C_Z^\ast
\end{bmatrix}.
}
```

At low angle of attack in normal upright flight, lift is upward, which is
approximately the negative body-$z$ direction. Therefore a normal positive lift
coefficient produces a negative $F_z$, consistent with $z_b$ being positive down.

---



## 11. Gravity

In NED coordinates, gravity is especially simple:

```math
\mathbf g^n=
\begin{bmatrix}0\\0\\g\end{bmatrix},
\qquad
\mathbf F_g^n=
\begin{bmatrix}0\\0\\mg\end{bmatrix}.
```

The force summation is performed in body axes, so gravity must first be rotated from
NED into body coordinates:

```math
\mathbf F_g^b
=
R_n^b\mathbf F_g^n
=
(R_b^n)^T
\begin{bmatrix}0\\0\\mg\end{bmatrix}.
```

Carrying out the multiplication gives

```math
\boxed{
\mathbf F_g^b=
\begin{bmatrix}
-mg\sin\theta\\
mg\cos\theta\sin\phi\\
mg\cos\theta\cos\phi
\end{bmatrix}.
}
```

That is exactly the `gravity` array in `ForcesAndMoments.calculate()`.

Examples:

- Level attitude: $\mathbf F_g^b=[0,0,mg]^T$, straight through the belly.
- Positive pitch: gravity has a negative $x_b$ component, pulling backward.
- Positive roll: gravity has a positive $y_b$ component, toward the right wing.

Gravity is the second important rotation in the force model: **gravity starts in NED
and is rotated into body coordinates so it can be added to the aerodynamic and
propulsive body-axis forces**.

---



## 12. Propulsion force

The propeller model produces thrust along the positive body $x_b$ axis:

```math
\boxed{
\mathbf F_p^b
=
\frac{1}{2}\rho S_{\text{prop}}C_{\text{prop}}
\begin{bmatrix}
(k_{\text{motor}}\delta_t)^2-V_a^2\\
0\\
0
\end{bmatrix}.
}
```

The first term represents propeller-induced flow or commanded propeller speed, while
the $-V_a^2$ term reduces net thrust as forward airspeed increases.

This simplified model includes:

- no propeller side force,
- no propeller normal force,
- no thrust-vector angle,
- no explicit thrust-line offset from the CG,
- a separate reaction-torque term in roll.

---



## 13. Total force

The returned force is

```math
\boxed{
\mathbf F_{\text{total}}^b
=
\mathbf F_g^b+\mathbf F_a^b+\mathbf F_p^b.
}
```

Every vector on the right-hand side has been expressed in body coordinates before
addition. This is essential: vectors may only be added component by component when
their components are resolved in the same frame.

### Converting the returned force to NED

If a logger, visualization, controller, or analysis routine needs the net force in
NED, calculate

```python
R_b_to_n = body_to_inertial(phi, theta, psi)
force_ned = R_b_to_n @ forces_body
acceleration_ned = force_ned / params.m
```

Since `forces_body` includes gravity:

```math
\mathbf a_{\text{NED,total}}
=
\frac{\mathbf F_{\text{NED,total}}}{m}.
```

If only the non-gravitational or **specific force** is required, rotate and divide
only the aerodynamic-plus-propulsive force:

```math
\mathbf f_{\text{specific}}^b
=
\frac{\mathbf F_a^b+\mathbf F_p^b}{m}.
```

An ideal accelerometer mounted with its sensing axes aligned to the body axes measures
specific force rather than gravity-inclusive inertial acceleration. This distinction
is important when comparing the simulation with IMU data.

---



## 14. Aerodynamic and propulsive moments

The returned moment vector is resolved about the body axes:

```math
\mathbf M^b=
\begin{bmatrix}
\ell\\m\\n
\end{bmatrix},
```

where

- $\ell$ is rolling moment about $+x_b$,
- $m$ is pitching moment about $+y_b$,
- $n$ is yawing moment about $+z_b$.

The nondimensional moment coefficients are

```math
\begin{aligned}
C_\ell^\ast &=
C_{\ell_0}+C_{\ell_\beta}\beta
+C_{\ell_p}\frac{b}{2V_a}p
+C_{\ell_r}\frac{b}{2V_a}r
+C_{\ell_{\delta_a}}\delta_a
+C_{\ell_{\delta_r}}\delta_r,\\
C_m^\ast &=
C_{m_0}+C_{m_\alpha}\alpha
+C_{m_q}\frac{c}{2V_a}q
+C_{m_{\delta_e}}\delta_e,\\
C_n^\ast &=
C_{n_0}+C_{n_\beta}\beta
+C_{n_p}\frac{b}{2V_a}p
+C_{n_r}\frac{b}{2V_a}r
+C_{n_{\delta_a}}\delta_a
+C_{n_{\delta_r}}\delta_r.
\end{aligned}
```

The standard dimensional aerodynamic moment model is

```math
\mathbf M_a^b
=
\bar qS
\begin{bmatrix}
bC_\ell^\ast\\
cC_m^\ast\\
bC_n^\ast
\end{bmatrix}.
```

The span $b$ converts roll and yaw coefficients to moment dimensions; the chord
$c$ does the same for pitch.

### Propeller reaction torque audit

The code places the propeller reaction-torque term inside the vector that is multiplied
by $\bar qS$. As written, its roll contribution is

```math
\ell_{\text{prop,code}}
=
-\bar qS\,k_{T_p}(k_\Omega\delta_t)^2.
```

A commonly used dimensional form instead places propeller torque outside the
aerodynamic scaling:

```math
\ell
=
\bar qSbC_\ell^\ast
-k_{T_p}(k_\Omega\delta_t)^2.
```

The distinction matters because the current code makes propeller reaction torque go
to zero with dynamic pressure, even if the propeller is spinning. Check the intended
units and source of `k_t_p`. If it is a dimensional torque coefficient, the second
form is likely the intended implementation.

---



## 15. Euler rotational dynamics

The rigid-body angular-momentum equation written in body coordinates is

```math
\mathbf M^b
=
J\dot{\boldsymbol\omega}^{\,b}
+\boldsymbol\omega^b\times(J\boldsymbol\omega^b).
```

For an aircraft symmetric about the $x_b$-$z_b$ plane, the inertia matrix normally
has the form

```math
J=
\begin{bmatrix}
J_x&0&-J_{xz}\\
0&J_y&0\\
-J_{xz}&0&J_z
\end{bmatrix}.
```

Solving the angular-momentum equation for $\dot p,\dot q,\dot r$ yields the
gamma-parameter form used by the code:

```math
\begin{aligned}
\dot p &=
\Gamma_1pq-\Gamma_2qr+\Gamma_3\ell+\Gamma_4n,\\
\dot q &=
\Gamma_5pr-\Gamma_6(p^2-r^2)+\frac{m}{J_y},\\
\dot r &=
\Gamma_7pq-\Gamma_1qr+\Gamma_4\ell+\Gamma_8n.
\end{aligned}
```

Here the symbol $m$ in the middle equation denotes pitching moment, not aircraft
mass. In the Python code it is named `pitch_moment` to avoid that ambiguity.

With

```math
\Gamma=J_xJ_z-J_{xz}^2,
```

the standard constants are

```math
\begin{aligned}
\Gamma_1&=\frac{J_{xz}(J_x-J_y+J_z)}{\Gamma},&
\Gamma_2&=\frac{J_z(J_z-J_y)+J_{xz}^2}{\Gamma},\\
\Gamma_3&=\frac{J_z}{\Gamma},&
\Gamma_4&=\frac{J_{xz}}{\Gamma},\\
\Gamma_5&=\frac{J_z-J_x}{J_y},&
\Gamma_6&=\frac{J_{xz}}{J_y},\\
\Gamma_7&=\frac{(J_x-J_y)J_x+J_{xz}^2}{\Gamma},&
\Gamma_8&=\frac{J_x}{\Gamma}.
\end{aligned}
```

These definitions should match the `Params` class.

Moments are not rotated into NED before applying the rotational equations because
angular rates, inertia, and moments are all represented in body axes. This is the
natural frame for aircraft rigid-body rotational dynamics because the inertia tensor
is constant in body coordinates.

---



## 16. Euler-angle kinematics

Body angular rates are not equal to Euler-angle rates. The code uses

```math
\begin{bmatrix}
\dot\phi\\\dot\theta\\\dot\psi
\end{bmatrix}
=
\begin{bmatrix}
1&\sin\phi\tan\theta&\cos\phi\tan\theta\\
0&\cos\phi&-\sin\phi\\
0&\sin\phi/\cos\theta&\cos\phi/\cos\theta
\end{bmatrix}
\begin{bmatrix}
p\\q\\r
\end{bmatrix}.
```

This is consistent with the same 3-2-1 Euler convention used by $R_b^n$.

The matrix becomes singular when $\cos\theta=0$, or
$\theta=\pm90^\circ$. A quaternion attitude representation is preferable if the
aircraft must operate near vertical pitch.

---



## 17. Complete state derivative

The model can be summarized as

```math
\dot{\mathbf p}^{\,n}=R_b^n\mathbf v^b,
```

```math
\dot{\mathbf v}^{\,b}
=
\frac{1}{m}
\left(\mathbf F_g^b+\mathbf F_a^b+\mathbf F_p^b\right)
-\boldsymbol\omega^b\times\mathbf v^b,
```

```math
\dot{\boldsymbol\eta}
=
T(\phi,\theta)\boldsymbol\omega^b,
\qquad
\boldsymbol\eta=
\begin{bmatrix}\phi\\\theta\\\psi\end{bmatrix},
```

and

```math
\dot{\boldsymbol\omega}^{\,b}
=
J^{-1}
\left(
\mathbf M^b-\boldsymbol\omega^b\times J\boldsymbol\omega^b
\right).
```

The derivative order returned by `Dynamics.derivative()` is

```math
\dot{\mathbf x}=
\begin{bmatrix}
\dot p_n&\dot p_e&\dot p_d&
\dot u&\dot v&\dot w&
\dot\phi&\dot\theta&\dot\psi&
\dot p&\dot q&\dot r
\end{bmatrix}^T.
```

---



## 18. RK4 time integration

The simulator advances the nonlinear state equation

```math
\dot{\mathbf x}=f(\mathbf x,\boldsymbol\delta)
```

using classical fourth-order Runge-Kutta:

```math
\begin{aligned}
k_1 &= f(\mathbf x_k,\boldsymbol\delta_k),\\
k_2 &= f(\mathbf x_k+\tfrac{\Delta t}{2}k_1,\boldsymbol\delta_k),\\
k_3 &= f(\mathbf x_k+\tfrac{\Delta t}{2}k_2,\boldsymbol\delta_k),\\
k_4 &= f(\mathbf x_k+\Delta t\,k_3,\boldsymbol\delta_k),\\
\mathbf x_{k+1} &=
\mathbf x_k+\frac{\Delta t}{6}(k_1+2k_2+2k_3+k_4).
\end{aligned}
```

Controls remain constant over the integration step. The forces and moments are
recalculated at each intermediate RK4 state, which is important because they depend
nonlinearly on velocity, angle of attack, sideslip, attitude, and angular rates.

---



## 19. Frame-flow summary

```text
Gravity in NED                         Air-relative velocity in body
    [0, 0, mg]                                  [u, v, w]
         |                                           |
         | R_n^b = (R_b^n)^T                         | Va, alpha, beta
         v                                           v
Gravity in body          Aerodynamic force in body       Propulsion in body
     F_g^b                         F_a^b                       F_p^b
         \                            |                         /
          \___________________________|________________________/
                                      |
                                      v
                         Total body force F_total^b
                                      |
                    +-----------------+-----------------+
                    |                                   |
                    | body Newton equation              | rotate with R_b^n
                    v                                   v
       v_dot^b = F_total^b/m - omega x v^b   F_total^n = R_b^n F_total^b
                    |                                   |
                    v                                   v
             integrate [u,v,w]                 a^n = F_total^n/m

Body velocity [u,v,w] --rotate with R_b^n--> NED position rate [p_n_dot,p_e_dot,p_d_dot]
```

---



## 20. Recommended verification tests

These checks catch most frame, sign, and rotation errors.

### Test 1: Rotation orthogonality

For several attitudes, verify

```math
(R_b^n)^TR_b^n=I,
\qquad
\det(R_b^n)=+1.
```

### Test 2: Level gravity

Set $\phi=\theta=0$. Confirm

```math
\mathbf F_g^b=[0,0,mg]^T.
```

### Test 3: Gravity round trip

For arbitrary roll, pitch, and yaw, verify

```math
R_b^n\mathbf F_g^b=
\begin{bmatrix}0\\0\\mg\end{bmatrix}.
```

Yaw should not change the NED gravity vector.

### Test 4: Level forward motion

With zero attitude and $[u,v,w]=[V,0,0]$, confirm

```math
\dot{\mathbf p}^{\,n}=[V,0,0]^T.
```

### Test 5: Heading rotation

With level attitude, $\psi=90^\circ$, and $[u,v,w]=[V,0,0]$, confirm

```math
\dot{\mathbf p}^{\,n}\approx[0,V,0]^T.
```

The aircraft should travel East.

### Test 6: Force/acceleration equivalence

For an arbitrary state, compare

```math
\frac{1}{m}R_b^n\mathbf F^b
```

with

```math
R_b^n
\left(
\dot{\mathbf v}^{\,b}
+\boldsymbol\omega^b\times\mathbf v^b
\right).
```

They should agree to numerical precision.

### Test 7: Straight-and-level trim

At a valid trim point, expect approximately

```math
\dot u=\dot v=\dot w=\dot p=\dot q=\dot r=0.
```

The body-axis force is not generally zero at trim because the rotating-frame terms
can be nonzero during turning flight. In straight, wings-level, constant-velocity
flight, the net body force should be approximately zero.

### Test 8: Specific-force check

With the aircraft stationary and level, aerodynamic and propulsion forces zero:

- inertial acceleration from total force is $+g$ downward,
- ideal accelerometer specific force is zero in free fall,
- an aircraft supported at rest requires an upward support force and its
  accelerometer reports the corresponding non-gravitational specific force.

This test helps prevent confusion between total acceleration and IMU specific force.

---



## 21. Modeling assumptions and limitations

The current model assumes:

- a rigid aircraft with constant mass and inertia,
- flat-Earth NED coordinates,
- Euler-angle attitude,
- forces and moments evaluated about the center of gravity,
- stationary atmosphere unless wind-relative velocity is added elsewhere,
- constant aerodynamic coefficients except for their stated dependencies,
- no actuator dynamics or control-surface saturation in these two classes,
- no ground contact,
- no fuel burn or moving center of gravity,
- no explicit propeller gyroscopic moment,
- no explicit center-of-pressure or thrust-line lever arm,
- a simplified post-stall lift model,
- a simplified propeller thrust model.


---



## 22. Implementation checklist

Before treating simulation output as validated flight physics, confirm:

1. `body_to_inertial()` matches the $R_b^n$ matrix in this document.
2. `Params.state0` uses the documented state order.
3. `gamma_1` through `gamma_8` use the documented inertia convention and the same
   sign for $J_{xz}$.
4. Aerodynamic moment coefficients are referenced to the center of gravity.
5. All control-derivative signs match the physical control conventions.
6. `k_t_p` has units consistent with the location of the propeller torque term.
7. Air-relative rather than ground-relative velocity is used when wind is enabled.
8. Altitude is reported as $-p_d$, not $p_d$.
9. Total acceleration and accelerometer specific force are not confused.
10. Force vectors are never added until they are expressed in the same frame.

---



## 23. Central takeaway

The force calculation is internally consistent because gravity, aerodynamics, and
propulsion are all assembled in body coordinates. The velocity equations then use
the rotating-frame form of Newton's second law. The position equations separately
rotate body velocity into NED.

The key relationships are

```math
\boxed{
\mathbf F_{\text{total}}^b
=
\mathbf F_g^b+\mathbf F_a^b+\mathbf F_p^b
}
```

```math
\boxed{
\dot{\mathbf v}^{\,b}
=
\frac{\mathbf F_{\text{total}}^b}{m}
-\boldsymbol\omega^b\times\mathbf v^b
}
```

```math
\boxed{
\dot{\mathbf p}^{\,n}=R_b^n\mathbf v^b
}
```

and, whenever an inertial-frame result is needed,

```math
\boxed{
\mathbf F_{\text{total}}^n=R_b^n\mathbf F_{\text{total}}^b,
\qquad
\mathbf a^n=\frac{1}{m}\mathbf F_{\text{total}}^n.
}
```

These equations provide the full path from locally modeled aircraft forces to
inertial motion under Newton's laws.
